# T1 · Necesidades, inventario y diccionario

E1 · Diagnóstico del caso Aguas Andinas. Datos y metas sintéticos para docencia; no representan operaciones reales. Ejecutar desde `E1/` para que `import comun` funcione.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import comun

comun.verificar_hashes()
crudo = comun.cargar_lotes()
df = comun.parse_exploratorio(crudo)
print(f'{len(crudo):,} filas físicas; {crudo.medicion_id.nunique():,} mediciones distintas')

253,404 filas físicas; 252,640 mediciones distintas


## Decisión, audiencias y preguntas

**Decisión:** priorizar tres sectores sintéticos para inspección considerando continuidad, diferencia de volúmenes, energía y confiabilidad de medición. Las señales operacionales se distinguen de la incertidumbre del sensor. El análisis no permite inferir potabilidad, cumplimiento regulatorio, causalidad ni resultados reales.

| Audiencia | Decisión y nivel de detalle |
|---|---|
| Ejecutiva | Jefatura ficticia de operaciones y planificación: asignar tres inspecciones y recursos. Necesita comparación de sectores, prioridades, incertidumbre y efecto del filtro de cobertura. |
| Operacional | Equipo ficticio de terreno, telemetría y mantenimiento: definir qué revisar y qué evidencia levantar. Necesita detalle por sector y semana, continuidad con límites, balance de volúmenes, energía y alertas de medición. |

| Pregunta medible | Cálculo y período/población | Acción que puede cambiar |
|---|---|---|
| ¿Qué sectores podrían quedar bajo la meta docente de continuidad? | Continuidad ponderada por conexiones y límites suponiendo 0 o 60 min en horas desconocidas; comparar intervalo con 99,5%, ene–dic 2025 UTC por sector. | Priorizar sectores cuyo máximo esté bajo la meta; si el intervalo la cruza, levantar evidencia antes de calificarlos. |
| ¿Dónde es mayor la diferencia entre entrada y salida? | Σ(entrada − salida)/Σ entrada en m³, ene–dic 2025 UTC, lecturas con ambos volúmenes convertibles y denominador positivo; informar faltantes y casos salida > entrada. | Revisar balance y medición en los sectores destacados, sin atribuir una causa por asociación. |
| ¿Qué tres sectores combinan señales operacionales y medición confiable para inspección? | Ranking de continuidad, diferencia de volúmenes, kWh/m³ y costo CLP/m³ disponible, junto a calidad; ago–dic 2025 UTC por sector, con y sin filtro de cobertura mínima. | Asignar tres inspecciones y definir evidencia para distinguir condición operacional de incertidumbre de sensor. |

Cero minutos es una lectura observada; nulo es un campo sin dato en una fila existente; ausente es una sector-hora sin fila.

## Inventario de fuentes y exposición

El hecho tiene granularidad sector-hora UTC (`fecha` inicia la hora) y clave nominal `medicion_id`. Los lotes conservan sus cortes independientes. Las dimensiones no se concatenan con las observaciones. La tabla de metas complementaria sustituye a la fuente anterior; las tarifas solo cubren agosto–diciembre.

In [2]:
meta = comun.META
filas = []
for item in meta['files']:
    rel = item['archivo']
    path = comun.RAIZ / rel
    contenido = path.read_bytes().replace(bytes([13, 10]), bytes([10]))
    sha = hashlib.sha256(contenido).hexdigest()
    filas.append({'archivo': rel, 'filas': item['filas'], 'columnas': item['columnas'], 'sha256': sha})
inventario = pd.DataFrame(filas)
inventario

,archivo,filas,columnas,sha256
0,datos/actualizacion/telemetria_horaria.csv,22432,11,16c591d7956a5c31da8a3de8b0bf12e83347e5b17758b4...
1,datos/ampliacion/telemetria_horaria.csv,100000,11,0f95e0e2a8759837d4753703c70d01409b56331b38a827...
2,datos/base/telemetria_horaria.csv,130972,11,1e9edf220a71b49f3ce472f73f6e0939b492129fd9e7e7...
3,datos/complementarios/metas_2025_ancho.csv,30,16,2ecbf487d1c7d61e7d6982ee183aad171b4ffdc03ca800...
4,datos/complementarios/sectores_historial.csv,60,7,e5eb2475a96725c6db1804a1cb44b002abf9d078529595...
5,datos/complementarios/tarifas_energia.csv,150,3,5b972d095c557ffa7bf5ef58dd3f755141c797771e3c88...
6,datos/dimensiones/metas_ancho.csv,30,11,67ed256a92faa117e591ad3ce33aa0843f7d243ab15ce4...
7,datos/dimensiones/sectores.csv,30,5,ac9d03a7fbd69b17cb725bbb0507c6e8982150ffa0d715...


| Archivo | Rol, clave y granularidad | Período y corte UTC |
|---|---|---|
| `datos/base/telemetria_horaria.csv` | Hecho; `medicion_id`; sector-hora | Ene–jun 2025; 2025-07-08 |
| `datos/actualizacion/telemetria_horaria.csv` | Hecho; `medicion_id`; sector-hora | Jul 2025; 2025-08-08 |
| `datos/ampliacion/telemetria_horaria.csv` | Hecho; `medicion_id`; sector-hora | Ago–dic 2025; 2026-01-08 |
| `datos/dimensiones/sectores.csv` | Dimensión fuente; `sector_id`; una fila por sector | Catálogo estático original |
| `datos/dimensiones/metas_ancho.csv` | Dimensión fuente; sector + indicador; meses en columnas | Ene–jul; reemplazada analíticamente |
| `datos/complementarios/sectores_historial.csv` | Dimensión SCD; sector + inicio de vigencia | 60 filas, dos versiones por sector; corte 2025-10-01 |
| `datos/complementarios/tarifas_energia.csv` | Complementaria; sector + mes | Ago–dic; 150 filas |
| `datos/complementarios/metas_2025_ancho.csv` | Dimensión; sector + indicador, meses en columnas | Ene–dic; hacer melt y usar en lugar de la anterior |

**Filas físicas:** 130.972 base + 22.432 actualización + 100.000 ampliación = 253.404. **Entidades:** 30 sectores válidos. **Exposición calendario:** 130.320 sector-horas base (181 × 24 × 30), 22.320 actualización (31 × 24 × 30) y 110.160 ampliación (153 × 24 × 30); total 262.800 sector-horas. Filas físicas pueden exceder la exposición por repetición, sector desconocido o fecha inválida; la cobertura se calcula desde la grilla esperada. El paquete declara 10.000 sector-horas sin recepción en ago–dic.

**Procedencia:** paquete v2, UTF-8, zona UTC, semillas 20260924 y 20261004, fecha del paquete 2026-10-01. Empresa usada solo como contexto; datos 100% sintéticos. SHA-256 de cada uno de los ocho archivos: tabla generada arriba y comprobación contra `metadatos.json` al inicio.

## Diccionario y contratos

El diccionario revisado extiende las 75 filas de `diccionario_datos.csv` con tipo estadístico, escala, unidad, nulabilidad, condición de clave, contrato verificable y observaciones. Los identificadores no se promedian; `fecha` representa el evento de negocio y `cargado_en` la primera carga. `presion_bar` usa escala de razón porque tiene cero físico y las razones entre magnitudes son interpretables dentro del escenario sintético; no implica criterio sanitario. Las metas mensuales están en columnas y deben pasar a formato largo. `conexiones` es un peso con vigencia, no clientes nuevos por hora.

La generación preserva una fila por campo de la fuente. Se revisará el texto de contratos junto con T2, que medirá las reglas; cualquier nulo observado se cuenta según su significado y no se completa automáticamente.

In [5]:
diccionario = pd.read_csv(comun.RAIZ / 'E1/salidas/tablas/diccionario_revisado.csv', dtype='string')
print(f'{len(diccionario)} campos documentados; {diccionario.archivo.nunique()} archivos; columnas: {list(diccionario.columns)}')
diccionario.head(12)

75 campos documentados; 8 archivos; columnas: ['archivo', 'campo', 'definicion', 'tipo_estadistico', 'escala', 'unidad', 'nulabilidad', 'es_clave', 'contrato', 'observaciones']


,archivo,campo,definicion,tipo_estadistico,escala,unidad,nulabilidad,es_clave,contrato,observaciones
0,datos/dimensiones/sectores.csv,sector_id,"Sector ficticio, disjunto de los demás para po...",identificador,nominal,id de sector,No previsto; nulo indica defecto,Sí,Debe existir en el catálogo; no se promedia.,Debe existir en el catálogo; no se promedia.
1,datos/dimensiones/sectores.csv,nombre_sector,Nombre docente de sector; no ubica infraestruc...,categórica nominal,nominal,etiqueta,No previsto; nulo indica defecto,No,Nombre sintético; no identifica infraestructur...,Nombre sintético; no identifica infraestructur...
2,datos/dimensiones/sectores.csv,comuna_referencia,"Comuna como etiqueta de contexto, no área de c...",categórica nominal,nominal,etiqueta,No previsto; nulo indica defecto,No,Etiqueta de contexto; no implica ubicación real.,Etiqueta de contexto; no implica ubicación real.
3,datos/dimensiones/sectores.csv,conexiones,Número sintético de conexiones expuestas del s...,cuantitativa discreta,razón,conexiones,No previsto; nulo indica defecto,No,Entero positivo; peso vigente para ponderar co...,Entero positivo; peso vigente para ponderar co...
4,datos/dimensiones/sectores.csv,capacidad_m3_h,Capacidad horaria sintética para contrastar me...,cuantitativa continua,razón,m³/h,No previsto; nulo indica defecto,No,Capacidad sintética no negativa para evaluar m...,Capacidad sintética no negativa para evaluar m...
5,datos/dimensiones/metas_ancho.csv,sector_id,"Sector ficticio, disjunto de los demás para po...",identificador,nominal,id de sector,No previsto; nulo indica defecto,Sí,Debe existir en el catálogo; no se promedia.,Debe existir en el catálogo; no se promedia.
6,datos/dimensiones/metas_ancho.csv,indicador,Nombre del indicador cuya meta aparece en las ...,categórica nominal,nominal,categoría,No previsto; nulo indica defecto,Sí,Indicador definido por catálogo.,Indicador definido por catálogo.
7,datos/dimensiones/metas_ancho.csv,unidad,Unidad de la meta o umbral.,categórica nominal,nominal,categoría,No previsto; nulo indica defecto,No,Unidad declarada de la meta.,Unidad declarada de la meta.
8,datos/dimensiones/metas_ancho.csv,sentido,maximo indica límite superior; minimo indica p...,categórica nominal,nominal,categoría,No previsto; nulo indica defecto,No,mínimo o máximo según definición de meta.,mínimo o máximo según definición de meta.
9,datos/dimensiones/metas_ancho.csv,m2025_01,Meta docente para m2025 01,cuantitativa continua,razón,valor de meta,No previsto; validar meta definida para el mes,No,Meta numérica docente; mes representado por el...,Columna ancha; no representa observaciones ind...


## Límites de interpretación

1. Las metas de continuidad, capacidades, presiones y registros son supuestos docentes sintéticos; una comparación con la meta no acredita cumplimiento regulatorio ni desempeño real.
2. Las horas sin lectura producen un intervalo de continuidad, no un valor conocido; la concentración de faltantes puede cambiar el orden de prioridades.
3. Una diferencia entre entrada y salida o una asociación con presión/energía no identifica la causa; el modelo no incluye almacenamiento ni transferencias y no sustituye una inspección.
4. La tarifa existe solo para ago–dic y no se debe tratar su ausencia anterior como costo cero.

**Reproducibilidad:** hashes verificados contra `metadatos.json`; lectura original como texto y parseo exploratorio sin borrar, deduplicar ni imputar filas. La limpieza corresponde a E2.